# Artificial Neural Network

### Importing the libraries

In [0]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
tf.__version__

## Part 1 - Data Preprocessing

### Importing the dataset

In [0]:
dataset = pd.read_csv('Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values    
#the first 3 columns do not affect the output, as rollnumber, customer id and surname have no relation with whether the patient will leave or not
y = dataset.iloc[:, -1].values

In [4]:
print(X)

In [5]:
print(y)

### Encoding categorical data

Label Encoding the "Gender" column

In [0]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])
#Used for binary categories (like Yes/No or Male/Female) or ordinal data where order matters. It assigns each category a number (0, 1, 2...).

In [7]:
print(X)

One Hot Encoding the "Geography" column

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))
#Used for nominal categories where there is no relationship between them (like "France", "Spain", "Germany"). 
#It creates new binary columns for each category to avoid the model thinking one country is "greater" than another.
#For example, France=[1,0,0], Spain=[0,1,0], Germany=[0,0,1]

In [9]:
print(X) 

### Splitting the dataset into the Training set and Test set

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

### Feature Scaling

In [0]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Part 2 - Building the ANN

### Initializing the ANN

In [0]:
ann = tf.keras.models.Sequential()
#tf.keras: This refers to the Keras API integrated into TensorFlow. It is the most popular high-level library for building and training deep learning models because it is user-friendly and modular.
#models.Sequential(): This is a specific class in Keras used to create a linear stack of layers.

### Adding the input layer and the first hidden layer

In [0]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu', input_shape=(X_train.shape[1],)))
#Dense: This is a standard "fully connected" layer where every node in this layer connects to every node in the previous layer.
#units=6: This is the number of neurons in this layer. 6=(11+1)/2 11 independent input vfeatures and 1 output feature.
#ctivation='relu': ReLU stands for Rectified Linear Unit. It’s a mathematical function that helps the network learn complex patterns.

### Adding the second hidden layer

In [0]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))          #second hidden layer
#We add a second layer to make the model "deeper," allowing it to learn more intricate relationships in your data.

### Adding the output layer

In [0]:
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))  
#units=1: Since this is likely a binary classification (e.g., Yes/No, 0/1), we only need one output neuron.
#activation='sigmoid': In the final layer, Sigmoid squashes the output into a range between 0 and 1. This is perfect because it gives you the probability of the outcome

## Part 3 - Training the ANN

### Compiling the ANN

In [0]:
ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
#optimizer='adam': Think of the optimizer as the "driver" of the model. Adam (Adaptive Moment Estimation) is an algorithm that updates the weights of the neurons to reduce errors. It is widely used because it's very efficient and requires little tuning.
#loss='binary_crossentropy': This is the "ruler" used to measure how wrong the model is. For binary outcomes (0 or 1), we use binary cross-entropy. If you were predicting categories (like Dog vs. Cat vs. Bird), you’d use categorical_crossentropy.
#metrics=['accuracy']: This is simply how we, as humans, judge the model performance (the percentage of correct guesses).

### Training the ANN on the Training set

In [17]:
ann.fit(X_train, y_train, batch_size=32, epochs=100)
#batch_size=32: Instead of looking at every single row of data one by one, the model looks at 32 rows at a time before updating its internal weights. This makes training faster.
#epochs=100: This means the neural network will go through the entire dataset 100 times. We do this because the model learns a little bit more with each pass.

## Part 4 - Making the predictions and evaluating the model

### Predicting the result of a single observation

**Extra**

Use our ANN model to predict if the customer with the following informations will leave the bank: 

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: \$ 60000

Number of Products: 2

Does this customer have a credit card ? Yes

Is this customer an Active Member: Yes

Estimated Salary: \$ 50000

So, should we say goodbye to that customer ?

**Solution**

In [18]:
print(ann.predict(sc.transform([[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])) > 0.5)

### Predicting the Test set results

In [19]:
y_pred = ann.predict(X_test)

### Making the Confusion Matrix

In [20]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
#[TP  FN
# FP  TN]

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.2%}")

write down about precision recall f1-score, why is it better than just accuracy, what are some other interesting metrics u can find

# Precision
- precision = true posituves/(True postives + False positives)
- It tells us how many the model got right among all the true postives
- High precision means less false positives

# Recall
- recall = true posituves/(True postives + False negatives)
- It tells us out of all the true cases, how many the the model detected
- High recall means less false negatives

# F1-Score
- F1-score = 2*Precision*Recall/(Precision + Recall)
- It is the harmonic mean of precision and recall
- It penalizes extreme values, if one is very low, F1-score will be low

# Why is F1-Score better than just Accuracy?
- Consider a situation where out of 1000 cases, 990 are false and 10 are true
- Accuracy will be 99% and it will tell that the model is perfect
- Whereas, precision and recall are 0 and they tell that the model is useless
- Therefore, only accuracy won't tell us whether the model is able to recognize the rare cases not only the majority result

# Other Interesting Metrics
- Specificity (True Negative Rate): Measures the proportion of actual negatives that were correctly identified. It is the "Recall" for the negative class.
- ROC-AUC (Area Under the Receiver Operating Characteristic Curve): Measures the model's ability to distinguish between classes across all possible thresholds. A score of 1.0 is perfect; 0.5 is no better than a random guess.
- PR-AUC (Area Under the Precision-Recall Curve): Often better than ROC-AUC for highly imbalanced datasets, as it ignores True Negatives and focuses strictly on the performance of the minority (positive) class.
- MCC (Matthews Correlation Coefficient): Considered one of the best "all-around" metrics for binary classification because it takes into account all four quadrants of the confusion matrix and remains reliable even if classes are of very different sizes.
- Log Loss (Cross-Entropy Loss): Unlike the metrics above which look at hard "Yes/No" predictions, Log Loss evaluates the probability or confidence of the prediction. It heavily penalizes confident but wrong predictions.
- Cohen’s Kappa: Measures the agreement between the model's predictions and the actual labels, adjusted for the agreement that might happen by pure chance.